# LACT ROOT -> pylast -> visualize

这个 notebook 用于检查一条完整链路：先用 `run_corsika_trace` 跑一个 CORSIKA/EventIO/simtel 文件，生成 `lact_event_root_v1` ROOT 文件；再用本地 `pylast.io.LactEventSource` 读取；最后通过 `pylast.visualize` 画图。

假设当前工作目录是 `LACT_sim` 仓库根目录，并且 `external/pylast` 已经在同一个 ROOT 环境里编译/安装完成。

## 1. 编译 LACT_sim ROOT 输出

服务器上如果 `module load root` 报 `Unable to locate a modulefile`，但 `root-config --version` 可以输出版本，例如 `6.36.04`，说明 ROOT 已经在当前 shell 里可用，只是没有 modulefile。此时用 `root-config --prefix` 给 CMake 指路径即可。

In [ ]:
%%bash
set -euo pipefail

make hessio
cmake -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DLACT_ENABLE_ROOT=ON \
  -DCMAKE_PREFIX_PATH="$(root-config --prefix)"
cmake --build build --target run_corsika_trace -j"${SLURM_CPUS_PER_TASK:-8}"

## 2. 跑一个 CORSIKA/EventIO/simtel 文件

把下面的 `INPUT_EVENTIO` 改成你的输入文件路径。默认使用 full-response 无 NSB 配置，方便检查真实 detector response 链路；配置里默认只跑 `source.max_shower_events=1`。输出 ROOT 文件会写到 `run_logs/lact_root_full_response/lact_events.root`。

In [ ]:
from pathlib import Path

input_eventio = Path('/path/to/input.simtel.zst')
cfg = Path('configs/examples/corsika_lact_root_full_response.cfg')
root_file = Path('run_logs/lact_root_full_response/lact_events.root')
pylast_plot_dir = Path('run_logs/lact_root_full_response/pylast_visualize')

print('config:', cfg)
print('input:', input_eventio)
print('ROOT output:', root_file)

In [ ]:
import subprocess

subprocess.run(
    [
        './build/run_corsika_trace',
        str(cfg),
        str(input_eventio),
    ],
    check=True,
)

## 3. 用 pylast 读取 LACT ROOT

`LactEventSource` 会把 LACT_sim ROOT 中的积分 p.e. 图映射到 `event.dl0`，把随时间的 p.e. 序列映射到 `event.r1.waveform`。如果文件只有积分图，没有 `waveforms` 树，则 R1 会退化成单采样点 waveform，方便 pylast 后续接口仍然能跑通。

In [ ]:
from pylast.io import LactEventSource

source = LactEventSource(str(root_file), max_events=10)
event = source[0]

print('n_events:', len(source))
print('event_id:', event.event_id)
print('telescope ids:', source.subarray.get_ordered_telescope_ids())
print('DL0 tels:', event.dl0.get_ordered_tels())
print('R1 tels:', event.r1.get_ordered_tels())
print('energy:', event.simulation.shower.energy)
print('alt(rad):', event.simulation.shower.alt)
print('az(rad):', event.simulation.shower.az)

In [ ]:
tel_id = event.r1.get_ordered_tels()[0]
r1 = event.r1.tels[tel_id]
dl0 = event.dl0.tels[tel_id]

print('selected tel:', tel_id)
print('DL0 image shape:', dl0.image.shape, 'sum p.e.:', float(dl0.image.sum()))
print('DL0 peak_time shape:', dl0.peak_time.shape)
print('R1 waveform shape:', r1.waveform.shape, 'sum p.e.:', float(r1.waveform.sum()))

## 4. 用 pylast.visualize 画图

可以直接使用 `EventVisualizer`，也可以使用新增的 `plot_lact_root_quicklook` 一次性生成标准检查图。

In [ ]:
from pylast.visualize import EventVisualizer, plot_lact_root_quicklook

pylast_plot_dir.mkdir(parents=True, exist_ok=True)
visualizer = EventVisualizer(source)

visualizer.plot_gathered_event(
    event,
    image_level='dl0',
    output_path=str(pylast_plot_dir / 'event0_gathered_dl0.png'),
    show_hillas=False,
    show=True,
)

visualizer.plot_event(
    event,
    image_level='dl0',
    output_path=str(pylast_plot_dir / 'event0_cameras_dl0.png'),
    show_hillas=False,
    show=True,
)

visualizer.plot_telescopes(
    event,
    image_level='dl0',
    output_path=str(pylast_plot_dir / 'event0_telescopes_dl0.png'),
    show=True,
)

In [ ]:
result = plot_lact_root_quicklook(
    root_file=str(root_file),
    output_dir=str(pylast_plot_dir / 'quicklook'),
    event_index=0,
    image_level='dl0',
    show=True,
)

for name, path in result['paths'].items():
    print(name, path)

## 5. 可选：直接检查 ROOT 文件内容

这个脚本不经过 pylast，使用 `uproot` 直接检查 ROOT 树结构、积分图、peak time 和 waveform 总和，适合定位是 LACT_sim 写出问题，还是 pylast 映射问题。

In [ ]:
%%bash
set -euo pipefail
python/plot_lact_root_output.py \
  run_logs/lact_root_full_response/lact_events.root \
  --outdir run_logs/lact_root_full_response/root_quicklook